# 5.1 数据并行 (Data Parallelism)

数据并行是最基础的分布式训练方式，将数据批次分配到多个设备，各设备持有完整模型副本并行计算。

本节涵盖：
- 数据并行（DP）
- 分布式数据并行（DDP）
- 完全分片数据并行（FSDP）
- ZeRO优化

## 1. 数据并行（DP）与分布式数据并行（DDP）

**DP**：最基础的并行方式，每张卡持有完整模型副本，将数据批次均匀分配到各卡，各卡独立计算梯度后进行AllReduce同步梯度。

**DDP**：DP的改进版本，使用Ring-AllReduce进行梯度同步，通信量与卡数无关，是PyTorch原生的分布式数据并行方案。

**核心流程**：
1. 每张GPU持有完整模型副本
2. 数据批次均分到各GPU
3. 各GPU独立前向+反向传播
4. AllReduce同步梯度
5. 各GPU独立更新参数

**DP vs DDP**：
- DP：参数服务器模式，通信瓶颈在server
- DDP：Ring-AllReduce，无中心节点，通信效率更高

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

model = nn.Sequential(nn.Linear(1024, 1024), nn.ReLU(), nn.Linear(1024, 1024))
n_params = sum(p.numel() for p in model.parameters())
param_bytes = n_params * 4
optimizer_bytes = n_params * 4 * 2
grad_bytes = n_params * 4

print('=== Data Parallelism (DP/DDP) ===')
print(f'Model parameters: {n_params:,}')
print(f'\nMemory per GPU (FP32):')
print(f'  Model weights: {param_bytes/1e9:.2f} GB')
print(f'  Optimizer states (Adam): {optimizer_bytes/1e9:.2f} GB')
print(f'  Gradients: {grad_bytes/1e9:.2f} GB')
print(f'  Total per GPU: {(param_bytes+optimizer_bytes+grad_bytes)/1e9:.2f} GB')

for n_gpus in [1, 2, 4, 8]:
    total_mem = n_gpus * (param_bytes + optimizer_bytes + grad_bytes)
    throughput = n_gpus
    print(f'\n  {n_gpus} GPUs: total memory={total_mem/1e9:.1f} GB, throughput={throughput}x')

print(f'\nKey: Each GPU holds a FULL copy of model + optimizer + gradients.')
print(f'DDP uses Ring-AllReduce for efficient gradient synchronization.')
print(f'Limitation: Model must fit on a single GPU!')

## 2. 完全分片数据并行（FSDP）

**目的**：突破单卡显存瓶颈

**基本原理**：将模型参数、梯度和优化器状态分片存储在不同设备上，计算时按需聚合所需分片，计算完成后立即释放，使单卡仅需存储1/N的模型状态。

**FSDP工作流程**：
1. 每个GPU只存储1/N的参数分片
2. 计算某层时，AllGather聚合该层完整参数
3. 执行前向/反向计算
4. 计算完成后立即释放非本地分片
5. 反向传播时ReduceScatter梯度

**代表**：PyTorch FSDP，是ZeRO-3的生产级实现

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

model = nn.Sequential(nn.Linear(1024, 1024), nn.ReLU(), nn.Linear(1024, 1024))
n_params = sum(p.numel() for p in model.parameters())
param_bytes = n_params * 4
opt_bytes = n_params * 4 * 2
grad_bytes = n_params * 4
total_per_gpu_no_shard = param_bytes + opt_bytes + grad_bytes

print('=== FSDP (Fully Sharded Data Parallel) ===')
print(f'Model: {n_params:,} params')
print(f'\nWithout FSDP (each GPU needs):')
print(f'  {total_per_gpu_no_shard/1e9:.2f} GB')

print(f'\nWith FSDP (sharded across N GPUs):')
for n_gpus in [2, 4, 8, 16]:
    shard_params = param_bytes / n_gpus
    shard_opt = opt_bytes / n_gpus
    shard_grad = grad_bytes / n_gpus
    total_shard = shard_params + shard_opt + shard_grad
    print(f'  {n_gpus:2d} GPUs: {total_shard/1e9:.2f} GB per GPU (1/{n_gpus} of full)')

print(f'\nFSDP workflow per layer:')
print(f'  1. AllGather: collect full layer parameters from all shards')
print(f'  2. Compute: forward/backward pass with full parameters')
print(f'  3. Discard: release non-local parameter shards')
print(f'  4. ReduceScatter: shard and distribute gradients')
print(f'\nKey: FSDP enables training models larger than single GPU memory.')

## 3. ZeRO优化

**目的**：系统性地减少显存占用

**基本原理**：分三个阶段逐步将单卡显存从O(M)降至O(M/N)，M为模型大小，N为GPU数量。

**ZeRO三个阶段**：
- **ZeRO-1**：分片优化器状态（最大节省，优化器约占4倍模型大小）
- **ZeRO-2**：额外分片梯度
- **ZeRO-3**：额外分片模型参数（=FSDP）

**显存分析**（Adam优化器，FP32训练）：
- 模型参数：M bytes
- 梯度：M bytes
- 优化器状态：2M bytes（一阶动量+二阶动量）
- 总计：4M bytes per GPU without ZeRO

In [ ]:
import torch

model_params_b = 7
bytes_per_param = 4
model_gb = model_params_b * 1e9 * bytes_per_param / 1e9
grad_gb = model_gb
optimizer_gb = model_gb * 2
total_gb = model_gb + grad_gb + optimizer_gb

print('=== ZeRO Optimization Stages ===')
print(f'Model: {model_params_b}B params (~{model_gb:.0f} GB in FP32)')
print(f'\nMemory breakdown (no ZeRO):')
print(f'  Parameters: {model_gb:.0f} GB')
print(f'  Gradients:  {grad_gb:.0f} GB')
print(f'  Optimizer:  {optimizer_gb:.0f} GB (Adam: momentum + variance)')
print(f'  Total:      {total_gb:.0f} GB per GPU')

n_gpus = 8
print(f'\nWith {n_gpus} GPUs:')
for stage in range(4):
    if stage == 0:
        mem = total_gb
        label = 'No ZeRO'
    elif stage == 1:
        mem = model_gb + grad_gb + optimizer_gb / n_gpus
        label = 'ZeRO-1 (shard optimizer)'
    elif stage == 2:
        mem = model_gb + grad_gb / n_gpus + optimizer_gb / n_gpus
        label = 'ZeRO-2 (+ shard gradients)'
    else:
        mem = model_gb / n_gpus + grad_gb / n_gpus + optimizer_gb / n_gpus
        label = 'ZeRO-3 (+ shard parameters = FSDP)'
    print(f'  {label}: {mem:.1f} GB per GPU')

print(f'\nKey: ZeRO-1 gives the biggest win (optimizer is 50% of total memory).')
print(f'ZeRO-3/FSDP enables training models that exceed single GPU memory.')